# DIAMOND：Atari task 正常／扰动测试

在本机 VS Code Notebook 运行，GPU 作业通过 SSH 提交到 **YOUR_CLUSTER_HOST / RTX 5090 / Slurm**。
本 Notebook 不需要打开终端。先连接 UW VPN；点击 **Run All / 全部运行**，选择本项目 `.venv-notebook` Python 内核。
只在 Notebook 的隐藏密码输入框输入 UW 密码。不会修改 authorized_keys，也不会保存密码。

先跑 Breakout 冒烟测试（16 条 rollout）；通过后自动提交完整测试（1440 条 rollout）。
完整测试可能耗时数小时；保持 Notebook 运行。断开 Notebook 不会自动取消已提交作业。

|Task|成功阈值（100 决策步内）|终止规则|
|---|---|---|
|Breakout / score_once|≥1 个正奖励决策步|整局终止即停止，不重新开局|
|Pong / score_one_point|≥1 个正奖励决策步，即至少得到1分|同上；不要求赢整局|
|Boxing / score_five_times|≥5 个正奖励决策步|同上；不等于5拳|

这三个阈值是项目暂定值，完整测试前冻结。若100步导致成功率接近0，需要另设新的协议版本重新测所有条件，不能只改某个扰动条件。
每次预测1帧预处理观测，执行1个决策步；3次去噪不算3个环境步。每决策步最多4个原始ALE帧，100步名义上约6.67秒。


In [ ]:
from pathlib import Path
import csv, getpass, json, posixpath, shlex, time, uuid
import paramiko
from IPython.display import display, HTML, Image
from atari_protocol import TASKS, conditions
ROOT = Path.cwd()
assert (ROOT / 'diamond_atari_runner.py').exists(), '请在本项目目录打开 Notebook'
display(HTML('<b>任务定义（不是已测成功率）</b>'))
for game, task in TASKS.items():
    print(game, task)
print('条件:', conditions())


## 协议

- 每个游戏20个初始案例，seed=20261011…20261030；不同案例不同起点，同一案例复用相同ALE完整状态、预处理缓冲、随机数状态和策略记忆。
- 每个条件分别在 ALE 与 DIAMOND 世界模型自由运行100步以内，提前结束后剩余预算视为无奖励，不重开一局。
- 正常 + sham（完全重复正常条件，用轨迹哈希核对）+ 动作丢失 p=0.05/0.10/0.20/0.30/0.40 + 观测高斯噪声 σ=0.01/0.02/0.05/0.10/0.20。
- 动作丢失以 NOOP 替换执行动作，世界模型也接收实际执行动作；测试控制抗扰性。噪声在[0,1]尺度加到策略观测并裁剪；测试视觉输入抗扰性，不改变真实物理过程。
- 主指标：成功次数/20、Wilson 95%置信区间、相对正常成功率下降、配对bootstrap 95%区间。辅助指标：正奖励步数、符号奖励和、首次成功步、终止步；ALE额外记录原始得分。
- DIAMOND奖励头只预测−1/0/+1，因此不把预测奖励和当原始游戏分数。
- 训练总预算 Atari100k 不等于训练轨迹长度；训练轨迹长度及H/轨迹长度标为未知。条件窗口从checkpoint配置读取写入manifest。
- 两个域自由运行后可能选择不同动作，其成功率差只能作为诊断。预警信号需要独立的同动作预测误差和留出集校准，本版本不声称已实现预警。


In [ ]:
# 一次连接。人手输入密码；不会显示或写入输出。
ssh = paramiko.SSHClient()
known = Path.home() / '.ssh' / 'known_hosts_gix'
assert known.exists(), f'缺少已核验主机指纹文件: {known}'
ssh.load_host_keys(str(known))
ssh.set_missing_host_key_policy(paramiko.RejectPolicy())
secret = getpass.getpass('YOUR_NETID UW password (hidden): ')
try:
    ssh.connect('YOUR_CLUSTER_HOST', username='YOUR_NETID', password=secret,
                allow_agent=False, look_for_keys=False, timeout=15, auth_timeout=45, banner_timeout=20)
finally:
    del secret
ssh.get_transport().set_keepalive(30)
def remote(command, check=True):
    _, stdout, stderr = ssh.exec_command(command, timeout=60)
    out, err = stdout.read().decode(), stderr.read().decode()
    status = stdout.channel.recv_exit_status()
    if check and status:
        raise RuntimeError(f'Remote command failed ({status}): {err or out}')
    return out, err, status
print(remote('hostname; whoami')[0])


In [ ]:
# 上传到新建独立目录，不覆盖环境和旧实验。
run_id = time.strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:8]
run_root = '/mnt/qnap/YOUR_NETID/diamond-test/atari-robustness/' + run_id
remote('mkdir -p -- ' + shlex.quote(run_root))
sftp = ssh.open_sftp()
for name in ['diamond_atari_runner.py', 'atari_protocol.py', 'test_atari_protocol.py']:
    sftp.put(str(ROOT/name), run_root+'/'+name)
local_results = ROOT / 'atari-results' / run_id
local_results.mkdir(parents=True)
def record_status(**values):
    (local_results/'status.json').write_text(json.dumps(dict(run_root=run_root, **values), indent=2))
record_status(status='connected')
print('远程实验目录:', run_root)
print('本地结果目录:', local_results)


In [ ]:
def submit(profile):
    script = f"""#!/bin/bash
#SBATCH --partition=gpu
#SBATCH --gres=gpu:1
#SBATCH --nodelist=YOUR_SLURM_NODE
#SBATCH --cpus-per-task=2
#SBATCH --mem=8G
#SBATCH --time=04:00:00
#SBATCH --job-name=diamond-{profile}
#SBATCH --output={run_root}/{profile}-%j.out
#SBATCH --error={run_root}/{profile}-%j.err
set -euo pipefail
export HF_HOME=/mnt/qnap/YOUR_NETID/diamond-test/.cache/huggingface
export SDL_VIDEODRIVER=dummy
export OMP_NUM_THREADS=2
export CUBLAS_WORKSPACE_CONFIG=:4096:8
cd /mnt/qnap/YOUR_NETID/diamond-test/diamond
PY="$HOME/.local/share/diamond-test/envs/diamond-py310-cu128/bin/python"
"$PY" -m unittest discover -s {run_root} -p 'test_atari_protocol.py'
"$PY" {run_root}/diamond_atari_runner.py --profile {profile} --out {run_root}/{profile}
"""
    path = run_root + '/' + profile + '.slurm'
    with sftp.open(path, 'w') as f: f.write(script)
    out, _, _ = remote('sbatch --parsable ' + shlex.quote(path))
    job = out.strip().split(';')[0]
    assert job.isdigit(), out
    record_status(status='submitted', profile=profile, job=job)
    print(f'{profile}: 已提交 job {job}')
    return job

def wait_for_job(profile, job):
    last = None
    try:
        while True:
            queue, _, _ = remote('squeue -h -u YOUR_NETID -o ' + shlex.quote('%i %T %R'))
            state = '\n'.join(line for line in queue.splitlines() if line.split()[0] == job)
            if state != last:
                print(time.strftime('%H:%M:%S'), state.strip() or '作业已退出队列，核对输出')
                last = state
            record_status(status='monitoring', profile=profile, job=job, queue=state.strip())
            if not state.strip(): break
            time.sleep(10)
    except KeyboardInterrupt:
        print(f'监控已中断，Slurm job {job} 仍可能运行；可再次调用 wait_for_job。')
        raise
    # Accounting is disabled on this cluster; require a completion marker, never infer success from queue disappearance.
    _, _, status = remote('test -f ' + shlex.quote(run_root+'/'+profile+'/PASS'), check=False)
    if status:
        out, err, _ = remote('tail -n 50 ' + shlex.quote(run_root+f'/{profile}-{job}.out') + ' ' + shlex.quote(run_root+f'/{profile}-{job}.err'), check=False)
        record_status(status='failed', profile=profile, job=job)
        raise RuntimeError(out + '\n' + err)
    for name in sftp.listdir(run_root+'/'+profile):
        if name.endswith(('.json','.jsonl','.csv','.gif','.npz')) or name == 'PASS':
            target = local_results/profile
            target.mkdir(exist_ok=True)
            sftp.get(run_root+'/'+profile+'/'+name, str(target/name))
    record_status(status='passed', profile=profile, job=job)
    with (local_results/profile/'summary.csv').open(encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    headings = ['game','domain','condition','level','successes','cases','success_rate','ci95_low','ci95_high','drop_vs_normal']
    display(HTML('<table><tr>'+''.join('<th>'+x+'</th>' for x in headings)+'</tr>' + ''.join('<tr>'+''.join('<td>'+r[x]+'</td>' for x in headings)+'</tr>' for r in rows)+'</table>'))
    print('通过：所有案例完成，normal/sham轨迹哈希一致。')


In [ ]:
smoke_job = submit('smoke')
wait_for_job('smoke', smoke_job)


In [ ]:
# 前一单元失败时 Run All 会停止，不会继续提交完整测试。
assert (local_results/'smoke'/'PASS').exists(), '冒烟测试尚未通过'
full_job = submit('full')
wait_for_job('full', full_job)


## 结果解释

`summary.csv` 是实测数值，`trials.jsonl` 保存逐步奖励、期望动作和实际动作，`manifest.json` 保存版本、窗口、GPU与快照信息。
`normal` vs `drop/noise` 在同一domain内做配对比较；ALE vs WM用于检查预测域与模拟真值的差距。
不要把同一游戏的不同case当不同task；不要把不同游戏的原始分数直接平均。

完整实验总计3 tasks × 20 cases × 12 conditions × 2 domains = 1440条rollout，最多144000决策步。
20例只是首轮统计，低成功率时不确定性会很大。正式跨模型对比应共享同一case bank与任务阈值。

官方来源：[DIAMOND repo](https://github.com/eloialonso/diamond)、[supported Atari results](https://github.com/eloialonso/diamond/blob/main/results/data/DIAMOND.json)、[reward encoding](https://github.com/eloialonso/diamond/blob/main/src/models/rew_end_model.py)。
